# 10 세 버전의 실험 비교와 배포 전 검증

**상황:** “금요일 오후, 새 상담 AI를 배포해도 될까?” 3개 정상 질문과 재고 실패 질문 1개로 변경을 검증합니다.

**25분 핵심:** 1–7단계. Docker·모델은 수업 전에 준비합니다. 기본 실행은 모델 호출 총 24회(실패 요청 2 + 기준선 8 + 프롬프트 개선 8 + 기능 정책 6), 동시성 1입니다.

A 기준 프롬프트 / B 근거 제한 프롬프트 / C 근거 제한 + 기능 정책.
C의 실시간 조회 요청은 실제 서비스 정책으로 중단하며 모델 호출을 수행하지 않습니다. 가짜 모델 응답이 아닙니다.
같은 4개 사례와 같은 모델을 사용하고, 모델의 오판을 포함한 원본 결과를 보존합니다.

**수업 전 준비:** [Docker·VS Code 안내](../langfuse/README.md)의 명령을 터미널에서 마친 뒤 실행합니다.

```sh
python -m pip install -r requirements/week11-langfuse.txt
python week11/lab/langfuse/init_env.py
docker compose -f week11/lab/langfuse/compose.yaml up -d --wait --wait-timeout 300
ollama pull qwen2.5:1.5b-instruct
```

Ollama 앱과 Docker Desktop이 실행 중이어야 합니다. [Langfuse 화면](http://localhost:3011)에 접속하고
`week11/lab/langfuse/.env`의 `LANGFUSE_LOGIN_EMAIL`, `LANGFUSE_LOGIN_PASSWORD`로 로그인합니다.
이 파일은 본인 PC의 비공개 설정입니다. 노트북 출력이나 GitHub에 키·비밀번호를 복사하지 않습니다.

`.venv` Python 3.11 커널에서 위부터 Shift+Enter. 다른 Notebook의 변수는 필요 없습니다.
모델의 의미 오류는 관찰 대상이며, 연결 실패·파싱 오류를 가짜 성공으로 대체하지 않습니다.


**위에서부터 코드 셀을 세는 실행 지도**

VS Code 왼쪽 `[숫자]`는 실행 횟수입니다. 아래 번호는 설명 셀을 제외한 코드 셀 순서입니다. 첫 셀부터 한 개씩 실행하고 완료 후 다음 셀로 이동합니다.

| 코드 셀 | 단계 | 화면에서 누를 곳 | 확인 결과 |
|---|---|---|---|
| 1 | 환경 연결 | Langfuse의 Week 11 Ollama | 10번에서도 첫 코드 셀을 실행. 09번 변수에 의존하지 않음. |
| 2 | 1. 경계 사례 | 출력 ‘재고 질문 관찰 요청 열기’ → 생성 입력·출력 | 재고 질문의 실제 응답. 이번 실행이 맞았더라도 경계 사례로 유지. |
| 3 | 2. 평가 데이터 | Datasets → 셀에 출력된 week11-release-… 이름 | 정상 3개 + 재고 1개, 총 4개. Input과 Expected output을 구분. 기존 데이터의 source trace는 최초 등록 요청일 수 있음. |
| 4 | 3. A 기준선 | 출력 ‘A 기준선 실험 열기’ | 한 실험의 질문 4개와 실제 출력·지표. 다음 셀 실행 전 완료 확인. |
| 5 | 4. B 프롬프트 | 출력 ‘B 프롬프트 개선 실험 열기’ | 같은 모델·데이터로 지시만 변경. decision_b와 실패한 checks 확인. 결과가 항상 좋아지지는 않음. |
| 6 | 5. C 기능 정책 | 출력 ‘C 기능 정책 실험 열기’와 ‘C 재고 질문 요청 열기’ | Q24a의 capability-check·contract-checks 확인. 이 경로는 generation 없음, judge not_run. 정상 3건에는 실제 모델 호출. |
| 7 | 6. 비교와 판단 | A 실험 → Compare에서 이번 B/C 선택 → Display → Score matrix 또는 Grid | Score matrix로 structural_pass·false_accept·generation_ms, Grid로 실제 답변 확인. Notebook의 HOLD/REVIEW 이유와 대조. |
| 8 | 7. 서버 검증 | VS Code 저장 확인 표 / 연결된 trace | 3개 실험 × 4개 요청, 총 12개 저장 확인. C 재고 경로의 작업 수는 다른 요청보다 적은 것이 정상. |
| 9 | 8. 결과 저장 | VS Code → week11/lab/reports/langfuse/release-comparison.json | A/B/C 결과와 판단 기록 저장. 이어 python -m week11.lab.demo_guide로 이번 실행의 바로가기 생성. |

<details>
<summary>교수 설명 · 확인 질문</summary>

**학습 목적** — Notebook 간 숨은 상태 없이 비교 실험을 시작합니다.

**설명 예시** — 10번에서도 첫 셀을 실행합니다. 09번에서 만든 변수가 남아 있어야 동작하는 방식이면 다른 학생의 PC에서 재현하기 어렵습니다. 같은 코드가 필요한 연결과 데이터를 스스로 준비하도록 만든 구조를 확인합니다.

**학생 질문** — 09번을 실행했는데 왜 이 셀을 다시 실행하나요?

**기대 답변** — 파일별로 독립 실행해 커널의 숨은 상태에 의존하지 않게 하기 위해서입니다.

**오해·문제 해결** — 새 커널에서 위부터 실행합니다. 연결 실패를 임의의 mock 결과로 채우지 않습니다.

[전체 강의 준비 안내](https://smilesjcha.github.io/llmops-16week/week11/lecture/instructor/#cell-10-1)

</details>

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'week11/lab/langfuse_lab.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert sys.version_info[:2] == (3, 11), 'Python 3.11 .venv 커널을 선택하세요.'
import pandas as pd
from IPython.display import display, Markdown
from week11.lab.evaluation import load_data
from week11.lab.ollama_lab import OllamaClient
from week11.lab.langfuse_lab import connect_langfuse, traced_question, wait_for_trace
lf = connect_langfuse()  # 이 실습 폴더의 .env만 사용. 키를 출력하지 않음.
ollama = OllamaClient()
print(ollama.probe())
by_id = {q['id']: q for q in load_data()[1]}


## 1. 실패 관찰과 개선 가설

재고 질문의 실제 응답을 확인합니다. 단일 질문을 본 뒤 전체 성능이 좋아졌다고 판단하지 않습니다.

In [ ]:
from week11.lab.release_lab import ensure_dataset, run_version, release_decision, DEFAULT_POLICY
incident = traced_question(lf, ollama, by_id['Q24a'], prompt_version='baseline')
wait_for_trace(lf, incident)
display(Markdown(f"[재고 질문 관찰 요청 열기]({incident['url']})"))
print(incident['answer'])
print('가설 B: 근거 제한 프롬프트로 개선 / 가설 C: 미지원 기능을 서비스에서 차단')

**확인 결과 / 화면에서 볼 것**

실제 답변이 우연히 올바르더라도 경계 사례로 유지합니다. 실패 사례만 모으지 않고 정상 사례도 함께 넣습니다.

<details>
<summary>교수 설명 · 확인 질문</summary>

**학습 목적** — 관찰한 경계 사례에서 실험 가설을 세웁니다.

**설명 예시** — 먼저 재고 질문의 실제 기준선 응답을 보겠습니다. 이 한 번의 요청은 전체 성능 측정이 아니라 개선할 위험을 구체화하는 출발점입니다. 이번에 잘 답했다면 성공 사례로 그대로 기록하되, 현재 자료로 실시간 재고를 알 수 없다는 경계는 유지합니다.

**학생 질문** — 이번 한 번 맞았으면 회귀 데이터에서 빼도 될까요?

**기대 답변** — 아닙니다. 재발할 수 있는 중요한 경계라면 기대 행동을 정의해 유지합니다.

**오해·문제 해결** — 출력된 이번 trace 링크를 사용합니다. 과거 실패 화면을 이번 실행의 결과처럼 설명하지 않습니다.

[전체 강의 준비 안내](https://smilesjcha.github.io/llmops-16week/week11/lecture/instructor/#cell-10-2)

</details>

## 2. 정상 사례와 실패 사례의 고정 데이터

내용 해시가 같은 데이터만 재사용합니다. 다른 사람이 바꾼 평가 기준은 자동 덮어쓰지 않습니다.

In [ ]:
dataset = ensure_dataset(lf, incident)
display(pd.DataFrame([{'id':i.input['question_id'], 'question':i.input['question'],
                      'request_type':i.input['requested_capability'], 'expected':i.expected_output} for i in dataset.items]))
print('데이터:', dataset.name)
POLICY = dict(DEFAULT_POLICY)  # 실험 결과를 보기 전에 고정
print('배포 전 조건:', POLICY)

**확인 결과 / 화면에서 볼 것**

Datasets에 4개 사례가 보입니다. Q24a는 source trace와 연결됩니다. requested_capability는 상담 화면의 메뉴/라우트 입력입니다. 정답의 answerable 값으로 서비스 분기를 선택하지 않으며, 자유 문장의 의도 분류는 이 실습 범위 밖입니다.

<details>
<summary>교수 설명 · 확인 질문</summary>

**학습 목적** — 평가 데이터와 기준을 결과 보기 전에 고정합니다.

**설명 예시** — 정상 질문 세 개와 재고 질문 한 개가 들어갑니다. Input은 서비스 요청, Expected output은 판정할 기준입니다. 같은 내용의 데이터셋은 재사용하고 내용이 달라지면 버전으로 구분합니다. 재고 입력의 requested_capability는 명시된 기능 요청입니다. reference나 정답 라벨을 보고 C가 분기하는 것은 아닙니다.

**학생 질문** — 왜 실패 질문만 네 번 넣지 않고 정상 질문도 넣나요?

**기대 답변** — 실패를 고치는 변경이 정상 응답을 망가뜨리는지 함께 보기 위해서입니다.

**오해·문제 해결** — 네 문항은 회귀 확인용입니다. 결과를 본 뒤 통과하도록 지연 예산을 바꾸지 않습니다. 기존 source trace는 처음 등록한 요청일 수 있습니다.

[전체 강의 준비 안내](https://smilesjcha.github.io/llmops-16week/week11/lecture/instructor/#cell-10-3)

</details>

## 3. A 기준선의 실제 실험

같은 검색기와 모델을 사용합니다. 기본 동시성 1로 로컬 모델 큐 적체를 피합니다.

In [ ]:
baseline = run_version(lf, ollama, dataset, 'baseline')
display(Markdown(f"[A 기준선 실험 열기]({baseline['experiment_url']})"))
display(pd.DataFrame([{'id':r['question_id'], 'answer':r['output']['answer'],
                      'judge':r['output']['judge']} for r in baseline['rows']]))

**확인 결과 / 화면에서 볼 것**

Experiments에 기준선 실행 4개가 생깁니다. generation_ms, structural_pass, false_accept와 실제 답변을 함께 읽습니다.

<details>
<summary>교수 설명 · 확인 질문</summary>

**학습 목적** — A를 비교의 기준선으로 보존합니다.

**설명 예시** — A는 기본 프롬프트로 같은 네 질문을 실행합니다. 각 항목의 출력과 점수가 기록되는지 확인하고 완료된 뒤 B로 넘어갑니다. 기준선이 있어야 변경 후 결과가 단순히 좋거나 나쁜 것이 아니라 무엇이 나아졌고 무엇이 나빠졌는지 설명할 수 있습니다.

**학생 질문** — A 실행에 한 질문이 빠졌다면 B와 평균만 비교해도 될까요?

**기대 답변** — 안 됩니다. 같은 질문이 모두 실행되었는지부터 확인해야 합니다.

**오해·문제 해결** — 실험에 사용한 모델·데이터·프롬프트를 함께 보존합니다. A를 한 번 실행한 결과를 모집단 성능으로 일반화하지 않습니다.

[전체 강의 준비 안내](https://smilesjcha.github.io/llmops-16week/week11/lecture/instructor/#cell-10-4)

</details>

## 4. B 프롬프트 개선의 실제 실험

평가 데이터와 모델을 고정하고 프롬프트만 변경합니다. 성공을 가정하지 않습니다.

In [ ]:
candidate = run_version(lf, ollama, dataset, 'grounded')
display(Markdown(f"[B 프롬프트 개선 실험 열기]({candidate['experiment_url']})"))
decision_b = release_decision(baseline, candidate, POLICY)
print(decision_b['decision'])
display(pd.DataFrame(decision_b['checks']))

**확인 결과 / 화면에서 볼 것**

실제 실행에서 더 좋아질 수도, 비슷하거나 나빠질 수도 있습니다. 문장은 확인 불가인데 기권 필드가 틀린 사례를 찾아봅니다.

<details>
<summary>교수 설명 · 확인 질문</summary>

**학습 목적** — B는 프롬프트 변경만의 효과와 한계를 확인합니다.

**설명 예시** — B는 모델과 질문을 그대로 두고 근거 중심 지시로 바꿉니다. decision_b의 결론을 읽기 전에 실패한 checks를 보세요. 지시를 개선했는데도 재고를 수락했다면 말투 개선만으로 서비스 경계가 해결되지 않은 것입니다. B가 통과하더라도 네 사례의 이번 실행이라는 범위는 유지합니다.

**학생 질문** — 프롬프트를 더 자세히 쓰면 항상 좋아질까요?

**기대 답변** — 아닙니다. 결과와 회귀를 같은 데이터에서 확인해야 합니다.

**오해·문제 해결** — 기존 공개 예시는 HOLD였지만 학생 실행이 반드시 같아야 하는 것은 아닙니다. 실제 값과 사전 조건으로 설명합니다.

[전체 강의 준비 안내](https://smilesjcha.github.io/llmops-16week/week11/lecture/instructor/#cell-10-5)

</details>

## 5. C 기능 정책의 실제 적용

policy_qa만 구현된 서비스에서는 live_inventory를 조회할 수 없습니다. 서비스의 능력 범위를 코드로 제한합니다.

In [ ]:
from week11.lab.release_lab import capability_response
print('정책 안내:', capability_response('policy_qa'))
print('미지원 실시간 조회:', capability_response('live_inventory'))
guarded = run_version(lf, ollama, dataset, 'capability_guard')
display(Markdown(f"[C 기능 정책 실험 열기]({guarded['experiment_url']})"))
guard_case = next(row for row in guarded['rows'] if row['question_id'] == 'Q24a')
display(Markdown(f"[C 재고 질문 요청 열기]({guard_case['output']['url']})"))
decision_c = release_decision(baseline, guarded, POLICY)
print(decision_c['decision'])
display(pd.DataFrame(decision_c['checks']))

**확인 결과 / 화면에서 볼 것**

Q24a trace에는 capability-check와 contract-checks가 보이고 모델 generation은 없습니다. 정책 응답임을 명시합니다. 정상 질문 3개는 여전히 실제 Ollama 생성·평가를 실행합니다.

<details>
<summary>교수 설명 · 확인 질문</summary>

**학습 목적** — C는 모델 능력 대신 서비스의 지원 범위를 설계한 변경입니다.

**설명 예시** — C는 B에 기능 확인을 더합니다. policy_qa는 실제 모델로 답하고 live_inventory는 지원하지 않으므로 생성 전에 기권합니다. 재고 trace에는 capability-check와 contract-checks가 있으며 generation이 없습니다. 가짜 모델 답변을 만든 것이 아니라 모델을 호출하지 않는 실제 서비스 경로입니다.

**학생 질문** — C의 재고 답변이 빠른 이유를 “모델 최적화”라고 해도 될까요?

**기대 답변** — 아닙니다. 모델 호출을 생략한 정책 경로입니다. 정상 질문은 여전히 실제 모델을 호출합니다.

**오해·문제 해결** — 이 실습은 자유 문장의 의도를 자동 분류하지 않습니다. requested_capability가 입력으로 주어집니다. 이를 실제 제품에 적용하면 라우팅 정확도도 별도로 평가해야 합니다.

[전체 강의 준비 안내](https://smilesjcha.github.io/llmops-16week/week11/lecture/instructor/#cell-10-6)

</details>

## 6. 한 화면에서 비교하고 배포 보류·검토 결정

평균 점수가 필수 실패를 상쇄하지 않도록 개별 조건을 확인합니다. 4건으로 p95나 일반 성능을 주장하지 않습니다.

In [ ]:
display(pd.DataFrame([
 {'version':'A 기준선', **decision_b['summary']['baseline']},
 {'version':'B 프롬프트', **decision_b['summary']['candidate']},
 {'version':'C 기능 정책', **decision_c['summary']['candidate']},
]))
for label,decision in [('B',decision_b),('C',decision_c)]:
    print(label,decision['decision'],decision['reason'])

**확인 결과 / 화면에서 볼 것**

HOLD는 배포 보류, REVIEW는 사람 검토 대상으로 이동입니다. REVIEW도 자동 배포가 아닙니다. 인용·기권 검사 통과율은 답변 의미 정확도와 다릅니다. 작은 모델의 판정은 필수 실패를 덮어쓰지 않습니다.

<details>
<summary>교수 설명 · 확인 질문</summary>

**학습 목적** — 비교표를 읽고 HOLD와 REVIEW의 근거를 설명합니다.

**설명 예시** — Compare에서 이번 A/B/C를 함께 엽니다. Score matrix로 차이를 찾고 Grid로 답변을 읽습니다. HOLD는 자동 조건이 하나라도 미충족된 보류, REVIEW는 자동 조건을 통과해 사람 검토로 넘긴 상태입니다. 두 상태는 우리가 작성한 Python 정책의 출력입니다. Langfuse가 배포를 승인해 주는 기능이 아닙니다.

**학생 질문** — C가 구조 검사 100%이고 REVIEW면 배포가 끝난 것일까요?

**기대 답변** — 아닙니다. 작은 회귀 세트의 계약 검사만 통과했고 의미 검토·추가 사례·운영 판단이 남아 있습니다.

**오해·문제 해결** — false_accept는 답변 불가 한 문항에만 적용합니다. 정상 세 문항의 빈 점수를 0으로 채우지 않습니다. HOLD 원인은 지연·누락·형식 등일 수도 있으므로 checks를 읽습니다.

[전체 강의 준비 안내](https://smilesjcha.github.io/llmops-16week/week11/lecture/instructor/#cell-10-7)

</details>

## 7. 서버에 저장된 실험과 점수 검증

반환된 Python 객체만 믿지 않고 실제 Langfuse API에서 각 요청을 재조회합니다.

In [ ]:
checks = []
for experiment in [baseline,candidate,guarded]:
    for row in experiment['rows']:
        saved = wait_for_trace(lf,row['output'])
        checks.append({'version':experiment['version'],'id':row['question_id'],
                       'observations':len(saved['observations']),'scores':len(saved['scores'])})
display(pd.DataFrame(checks))
print('3개 실험 × 4개 요청의 서버 저장 확인')

**확인 결과 / 화면에서 볼 것**

Experiments에서 같은 데이터의 A/B/C를 선택하여 비교하고, 실패 셀에서 개별 trace로 이동합니다. 서비스 기능 정책의 경로는 generation이 없는 것이 정상입니다.

<details>
<summary>교수 설명 · 확인 질문</summary>

**학습 목적** — 보이는 점수뿐 아니라 모든 실행의 저장 완결성을 확인합니다.

**설명 예시** — 세 버전 곱하기 네 질문, 총 열두 요청이 서버에 저장됐는지 확인합니다. 빠진 요청을 제외하고 좋은 결과만 평균 내면 잘못된 판단이 됩니다. C의 재고 경로는 모델을 호출하지 않으므로 작업 수가 적고 judge가 not_run인 것이 정상입니다.

**학생 질문** — 모델 평가가 not_run이면 평가 실패인가요?

**기대 답변** — C의 정책 경로에서는 의도된 미실행입니다. 다른 경로의 오류나 누락과 구분해야 합니다.

**오해·문제 해결** — 비동기 저장이 늦으면 API 확인과 같은 링크 새로고침을 먼저 합니다. 모든 trace에 작업 수 5개를 강제하지 않습니다.

[전체 강의 준비 안내](https://smilesjcha.github.io/llmops-16week/week11/lecture/instructor/#cell-10-8)

</details>

## 8. 배포 판단 근거와 다음 실험 기록

관찰 → 실패 사례 → 고정 평가 → 변경 비교 → 배포 조건 → 사람 검토까지 연결합니다.

In [ ]:
out = ROOT/'week11/lab/reports/langfuse'
out.mkdir(parents=True,exist_ok=True)
report = {'baseline':baseline,'candidate':candidate,'guarded':guarded,'decision':decision_b,'guarded_decision':decision_c}
(out/'release-comparison.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(out/'release-comparison.json')
print('다음 과제: 정상 답변의 조건 누락을 사람이 검토하고, 검토 근거와 다음 변경 하나를 제안')
lf.flush()

**확인 결과 / 화면에서 볼 것**

09번도 기록 저장을 마쳤다면 터미널에서 `python -m week11.lab.demo_guide`로 이번 실행의 화면별 바로가기를 만듭니다. `reports/langfuse/instructor-console.html`을 브라우저로 여세요. 새 마감 과제가 아닌 수업 활동입니다. CLI의 --enforce-gate는 B가 HOLD이면 종료 코드 2로 자동 차단합니다. 실제 운영 배포·롤백은 실행하지 않습니다. 3초 예산은 교육용 기본값이며 기기·서비스 조건에 맞게 실험 전에 정해야 합니다.

<details>
<summary>교수 설명 · 확인 질문</summary>

**학습 목적** — 판단의 근거와 다음 실험을 검토 가능한 결과물로 남깁니다.

**설명 예시** — 세 실험의 원본 결과와 판단을 JSON으로 저장합니다. 이어 demo_guide를 실행하면 이번 실행의 링크로 바로 이동할 수 있습니다. 마지막 설명은 “어떤 버전을, 무슨 근거로, 어떤 상태로 두고, 다음에 무엇을 확인할지” 네 가지면 충분합니다.

**학생 질문** — 최종 보고에 가장 먼저 쓸 한 문장은 무엇일까요?

**기대 답변** — 예: C는 자동 계약 검사를 통과해 REVIEW이며, 실제 답변 의미 검토와 경계 사례 확장이 남아 있습니다.

**오해·문제 해결** — 실제 결과가 HOLD이면 그 원인으로 문장을 바꿉니다. 공개 예시 결과를 자신의 실행 결과로 복사하지 않습니다.

[전체 강의 준비 안내](https://smilesjcha.github.io/llmops-16week/week11/lecture/instructor/#cell-10-9)

</details>

## 마무리

연결이 끝나면 `lf.flush()`로 전송합니다. 실제 저장은 API 재조회로 확인합니다. 모델·프롬프트·기기 상태에 따라 답변과 지연은 달라집니다.
수업 결과는 합성 사례이며 일반적인 서비스 정확도 또는 운영 배포 승인이 아닙니다.

Docker 중지(데이터 유지): `docker compose -f week11/lab/langfuse/compose.yaml stop`
재개: 같은 경로의 `up -d --wait`. **`down -v`는 데이터를 삭제하므로 일반 종료에 사용하지 않습니다.**
[화면별 시연 안내](https://smilesjcha.github.io/llmops-16week/week11/lab/demo/) / [100분 진행·문제 해결](../langfuse/README.md) / [전체 실행 순서](../WALKTHROUGH.md)
